# Loss Run Analysis Notebook

Interactive analysis of training loss runs from CSV log files for PIDU, DU, and CNN models.

**Structure:**
1. Imports and Configuration
2. Mock Data Generation
3. Data Loading Utilities
4. Unit Tests for Data Loading and Parsing
5. Loss Curve Visualization
6. Train / Validation / Test Split Analysis
7. Per-Epoch Aggregation and Statistics
8. Model Comparison Utilities
9. Unit Tests for Aggregation and Comparison
10. Interactive Analysis Template

## 1. Imports and Configuration

In [ ]:
import sys
import os
import io
import unittest
import glob
import textwrap
import logging

import numpy as np
import pandas as pd
import torch
import jinja2
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


# Make project root importable when running from Analyse/
_PROJECT_ROOT = os.path.abspath(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
if _PROJECT_ROOT not in sys.path:
    sys.path.insert(0, _PROJECT_ROOT)

# Column schema
COLUMNS = ["dataset", "system", "batch", "epoch", "loss", "time"]
DATASET_TYPES = ["train", "val", "test", "exp"]

# Project-wide defaults
SKIPROWS_PIDU = 13   # header lines in PIDU / DU run files
SKIPROWS_CNN = 10     # header lines in CNN run files

# Path template
RUNS_ROOT = os.path.join(_PROJECT_ROOT, "casestudies")

print("Project root:", _PROJECT_ROOT)
print("NumPy:", np.__version__, "| Pandas:", pd.__version__, "| Pytorch:", torch.__version__)
print("Jinja2:", "jinja2" in sys.modules, "| Matplotlib:", matplotlib.__version__)
%matplotlib 
plt.rcParams.update({"figure.dpi": 50, "axes.grid": True, "grid.alpha": 0.3})
plt.rcParams["text.usetex"] = False

# Silence verbose font-manager messages and enforce portable serif fallback.
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["DejaVu Serif", "Times New Roman", "Times", "STIXGeneral"]
plt.rcParams["font.size"] = 8

## 2. Mock Data Generation

Synthetic DataFrames that mimic the real CSV format, so you can run every section below
without needing actual run files on disk.

In [ ]:
def make_mock_run(
    n_epochs: int = 10,
    n_train_batches: int = 64,
    n_val_batches: int = 4,
    n_test_batches: int = 4,
    n_exp_batches: int = 1,
    run_id: int = 0,
    seed: int = 0,
    start_loss: float = 90_000.0,
    decay: float = 0.75,
) -> pd.DataFrame:
    """Return a synthetic run DataFrame with realistic decaying loss."""
    rng = np.random.default_rng(seed)
    rows = []
    for epoch in range(n_epochs):
        scale = start_loss * (decay ** epoch)
        for ds, n_batches in [
            ("train", n_train_batches),
            ("val",   n_val_batches),
            ("test",  n_test_batches),
            ("exp",   n_exp_batches),
        ]:
            noise_factor = 0.15 if ds == "train" else 0.05
            for batch in range(n_batches):
                loss = scale * (1.0 + rng.normal(0, noise_factor))
                rows.append({
                    "dataset": ds,
                    "system": 0,
                    "batch": batch,
                    "epoch": epoch,
                    "loss": max(loss, 1.0),
                    "time": rng.uniform(0.02, 0.10),
                    "run_id": run_id,
                })
    return pd.DataFrame(rows)


# ── PIDU-style run: high initial loss, slow convergence ───────────────────────
mock_pidu_run0 = make_mock_run(
    n_epochs=10, start_loss=96_000.0, decay=0.78, run_id=0, seed=42
)
mock_pidu_run1 = make_mock_run(
    n_epochs=10, start_loss=94_000.0, decay=0.80, run_id=1, seed=123
)

# ── CNN-style run: lower initial loss, faster convergence ─────────────────────
mock_cnn_run0 = make_mock_run(
    n_epochs=10, start_loss=86_000.0, decay=0.65, run_id=0, seed=7
)
mock_cnn_run1 = make_mock_run(
    n_epochs=10, start_loss=85_000.0, decay=0.67, run_id=1, seed=99
)

# ── DU-style run ──────────────────────────────────────────────────────────────
mock_du_run0 = make_mock_run(
    n_epochs=10, start_loss=92_000.0, decay=0.72, run_id=0, seed=55
)

# Combined multi-run DataFrames
mock_pidu_combined = pd.concat([mock_pidu_run0, mock_pidu_run1], ignore_index=True)
mock_cnn_combined  = pd.concat([mock_cnn_run0,  mock_cnn_run1],  ignore_index=True)

print("PIDU combined shape:", mock_pidu_combined.shape)
print("CNN  combined shape:", mock_cnn_combined.shape)
mock_pidu_combined.head()

In [ ]:
def make_mock_run_file(
    n_epochs: int = 5,
    n_train_batches: int = 64,
    n_val_batches: int = 4,
    n_test_batches: int = 4,
    n_exp_batches: int = 1,
    start_loss: float = 90_000.0,
    decay: float = 0.75,
    n_header_lines: int = 13,
    seed: int = 0,
) -> str:
    """Return a mock run file as a string (header + CSV body).

    The returned string can be written to disk or wrapped in io.StringIO
    for in-memory parsing.
    """
    header = "\n".join([f"Run information:"] + [f"-line{i}" for i in range(n_header_lines - 1)])
    df = make_mock_run(n_epochs=n_epochs, n_train_batches=n_train_batches,
                       n_val_batches=n_val_batches, n_test_batches=n_test_batches,
                       n_exp_batches=n_exp_batches, start_loss=start_loss,
                       decay=decay, seed=seed)
    body = df[COLUMNS].to_csv(index=False)
    return header + "\n" + body

sample_file_str = make_mock_run_file(n_epochs=3)
print(sample_file_str[:500])

## 3. Data Loading Utilities

Parsers for real `.txt` run files on disk.

In [ ]:
def load_run(filepath: str, skiprows: int = SKIPROWS_PIDU) -> pd.DataFrame:
    """Load a single run file, skipping the metadata header lines.

    Parameters
    ----------
    filepath : str
        Path to the .txt run file.
    skiprows : int
        Number of header lines before the CSV data (13 for PIDU/DU, 9 for CNN).

    Returns
    -------
    pd.DataFrame with columns: dataset, system, batch, epoch, loss, time
    """
    df = pd.read_csv(filepath, skiprows=skiprows)
    df.columns = [c.strip() for c in df.columns]
    # Sort by epoch then batch to handle any out-of-order rows (e.g. loss_PIDU_6_009.txt)
    df = df.sort_values(["epoch", "dataset", "batch"]).reset_index(drop=True)
    return df


def load_run_from_string(raw: str, skiprows: int = SKIPROWS_PIDU) -> pd.DataFrame:
    """Like load_run but from a string (useful for testing / in-memory mock files)."""
    lines = raw.splitlines(keepends=True)
    csv_part = "".join(lines[skiprows:])
    df = pd.read_csv(io.StringIO(csv_part))
    df.columns = [c.strip() for c in df.columns]
    df = df.sort_values(["epoch", "dataset", "batch"]).reset_index(drop=True)
    return df


def filter_dataset(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    """Return only rows for a given dataset split (train / val / test / exp)."""
    return df[df["dataset"] == dataset].copy()


def filter_epoch(df: pd.DataFrame, epoch: int) -> pd.DataFrame:
    """Return only rows for a specific epoch."""
    return df[df["epoch"] == epoch].copy()


def load_multiple_runs(
    base_pattern: str,
    trail: int,
    num_runs: int = 10,
    usecase: int = 1,
    skiprows: int = SKIPROWS_PIDU,
) -> tuple[pd.DataFrame, list[pd.DataFrame]]:
    """Load several run files matching the naming convention and return a combined DataFrame.

    File path: casestudies/case{usecase}/runs/{base_pattern}_{run_idx}_{trail:03d}.txt

    Returns
    -------
    combined_df : pd.DataFrame  (with run_id column)
    individual_dfs : list[pd.DataFrame]
    """
    individual_dfs = []
    for i in range(num_runs):
        path = os.path.join(
            RUNS_ROOT,
            f"case{usecase}",
            "runs",
            f"{base_pattern}_{i}_{trail:03d}.txt",
        )
        if not os.path.exists(path):
            print(f"  [skip] not found: {os.path.basename(path)}")
            continue
        df = load_run(path, skiprows=skiprows)
        df["run_id"] = i
        individual_dfs.append(df)

    if not individual_dfs:
        raise FileNotFoundError(f"No run files found for pattern '{base_pattern}' trail {trail:03d}")

    combined_df = pd.concat(individual_dfs, ignore_index=True)
    return combined_df, individual_dfs


# ── Quick demo with mock string ───────────────────────────────────────────────
_demo_str = make_mock_run_file(n_epochs=3, n_header_lines=13)
_demo_df  = load_run_from_string(_demo_str, skiprows=13)
print("Loaded columns:", list(_demo_df.columns))
print("Datasets:", _demo_df["dataset"].unique())
_demo_df.head()

## 4. Unit Tests — Data Loading and Parsing

In [ ]:
class TestLoadRun(unittest.TestCase):

    def setUp(self):
        raw = make_mock_run_file(n_epochs=3, n_train_batches=8,
                                 n_val_batches=2, n_test_batches=2,
                                 n_exp_batches=1, n_header_lines=13, seed=0)
        self.df = load_run_from_string(raw, skiprows=13)

    def test_columns_present(self):
        for col in COLUMNS:
            self.assertIn(col, self.df.columns)

    def test_datasets_present(self):
        found = set(self.df["dataset"].unique())
        self.assertEqual(found, {"train", "val", "test", "exp"})

    def test_epoch_count(self):
        epochs = sorted(self.df["epoch"].unique())
        self.assertEqual(epochs, [0, 1, 2])

    def test_train_batch_count_per_epoch(self):
        train_e0 = filter_dataset(filter_epoch(self.df, 0), "train")
        self.assertEqual(len(train_e0), 8)

    def test_filter_dataset_returns_correct_rows(self):
        val_df = filter_dataset(self.df, "val")
        self.assertTrue((val_df["dataset"] == "val").all())

    def test_filter_epoch_returns_correct_rows(self):
        e1 = filter_epoch(self.df, 1)
        self.assertTrue((e1["epoch"] == 1).all())

    def test_out_of_order_batches_sorted(self):
        """Rows must be sorted by (epoch, dataset, batch) after loading."""
        epochs = self.df["epoch"].tolist()
        self.assertEqual(epochs, sorted(epochs))

    def test_skiprows_9_cnn(self):
        raw = make_mock_run_file(n_epochs=2, n_header_lines=9)
        df  = load_run_from_string(raw, skiprows=9)
        self.assertIn("loss", df.columns)


suite = unittest.TestLoader().loadTestsFromTestCase(TestLoadRun)
runner = unittest.TextTestRunner(verbosity=2)
runner.run(suite)

## 5. Loss Curve Visualization

Plot per-epoch mean loss for each dataset split, with optional shaded std bands
and support for overlaying multiple runs.

In [ ]:
from tkinter import font
from turtle import title


def plot_loss_curves(
    dfs: dict[str, pd.DataFrame],
    splits: list[str] | None = None,
    log_scale: bool = False,
    shade_std: bool = True,
    figsize: tuple = (24, 10),
    style: list[str] | None = None,
    **kwargs,
) -> tuple:
    """Plot per-epoch mean loss curves for one or more labelled runs.

    Parameters
    ----------
    dfs : dict  {label: combined_df}  - each df must have 'dataset', 'epoch', 'loss' columns
    splits : list of dataset splits to plot (default: ['train', 'val'])
    log_scale : plot y-axis on log scale
    shade_std : overlay translucent +/-1 sigma band

    Returns
    -------
    fig, axes  (one subplot per split)
    """
    if splits is None:
        splits = ["train", "val"]

    if style:
        plt.style.use(style)

    # Some styles (e.g. science/ieee) enable external LaTeX and Times-only serif.
    plt.rcParams["text.usetex"] = False
    plt.rcParams["font.family"] = "serif"
    plt.rcParams["font.serif"] = ["DejaVu Serif", "Times New Roman", "Times", "STIXGeneral"]

    fig, axes = plt.subplots(1, len(splits), figsize=figsize, sharey=False)
    if len(splits) == 1:
        axes = [axes]

    # colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    colors = plt.cm.tab10.colors  # use a fixed color palette for consistency

    for ax, split in zip(axes, splits):
        for idx, (label, df) in enumerate(dfs.items()):
            sub_df = df[df["dataset"] == split].copy()
            if sub_df.empty:
                continue

            # Coerce to numeric to avoid object dtype arrays that break matplotlib.fill_between.
            sub_df["epoch"] = pd.to_numeric(sub_df["epoch"], errors="coerce")
            sub_df["loss"] = pd.to_numeric(sub_df["loss"], errors="coerce")
            sub_df = sub_df.dropna(subset=["epoch", "loss"])
            if sub_df.empty:
                continue

            grouped = sub_df.groupby("epoch")["loss"]
            mean = grouped.mean().sort_index()
            std = grouped.std().fillna(0).reindex(mean.index, fill_value=0)

            x = mean.index.to_numpy(dtype=np.float64)
            y = mean.to_numpy(dtype=np.float64)
            y_std = std.to_numpy(dtype=np.float64)

            finite = np.isfinite(x) & np.isfinite(y) & np.isfinite(y_std)
            if not np.any(finite):
                continue

            x = x[finite]
            y = y[finite]
            y_std = y_std[finite]

            c = colors[idx % len(colors)]
            ax.plot(x, y, label=label, color=c)
            if shade_std:
                ax.fill_between(
                    x,
                    y - y_std,
                    y + y_std,
                    alpha=0.15,
                    color=c,
                )

        ax.set(
            xlim=kwargs.get("xlim", (0, None)),
            ylim=kwargs.get("ylim", None),
            xscale=kwargs.get("xscale", "linear"),
            yscale=kwargs.get("yscale", "linear"),
            title=split,
            xlabel="Epoch",
            ylabel="Mean Loss",
            # fontsize=kwargs.get("fontsize", 4)
        )
        ax.legend(fontsize=kwargs.get("fontsize_legend", 2))

    fig.suptitle("Loss Curves", fontweight="bold")
    fig.tight_layout()
    fig.subplots_adjust(left=0.05, right=0.98, wspace=0.25)
    return fig, axes

In [ ]:
# ── Example: compare PIDU vs CNN on train + val ────────────────────────────────
fig, axes = plot_loss_curves(
    dfs={"PIDU": mock_pidu_combined, "CNN": mock_cnn_combined},
    splits=["train", "val"],
    shade_std=True,
)
plt.show()

# Log-scale version
fig2, _ = plot_loss_curves(
    dfs={"PIDU": mock_pidu_combined, "CNN": mock_cnn_combined},
    splits=["train", "val"],
    shade_std=False,
)
plt.show()

In [ ]:
def plot_all_splits(df: pd.DataFrame, title: str = "Run", log_scale: bool = False, **kwargs) -> tuple:
    """Four-panel plot: train / val / test / exp loss curves for a single combined df."""
    fig, axes = plt.subplots(1, 4, figsize=(24, 10), sharey=False)
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for ax, split in zip(axes, DATASET_TYPES):
        sub = df[df["dataset"] == split]
        if sub.empty:
            ax.set_title(f"{split} (no data)")
            continue

        if "run_id" in sub.columns:
            for i, (rid, rdf) in enumerate(sub.groupby("run_id")):
                mean = rdf.groupby("epoch")["loss"].mean()
                ax.plot(mean.index, mean.values,
                        color=colors[i % len(colors)], alpha=0.6,
                        label=f"run {rid}")

        # Global mean across runs
        gmean = sub.groupby("epoch")["loss"].mean()
        ax.plot(gmean.index, gmean.values, "k--", lw=2, label="mean")

        if log_scale:
            ax.set_yscale("log")
        ax.set_title(split)
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.legend(fontsize=kwargs.get("fontsize_legend", 2))

    fig.suptitle(title, fontweight="bold")
    fig.tight_layout()
    return fig, axes


fig, _ = plot_all_splits(mock_pidu_combined, title="PIDU — all splits", log_scale=False)
plt.show()

## 6. Train / Validation / Test Split Analysis

Summary tables and best-epoch identification from per-epoch statistics.

In [ ]:
def split_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Return a pivot table: rows=epoch, columns=dataset, values=mean loss."""
    summary = (
        df.groupby(["epoch", "dataset"])["loss"]
        .mean()
        .unstack("dataset")
        .rename_axis(None, axis=1)
    )
    return summary


def best_val_epoch(df: pd.DataFrame) -> int:
    """Return the epoch with the minimum mean validation loss."""
    val = df[df["dataset"] == "val"].groupby("epoch")["loss"].mean()
    return int(val.idxmin())


pidu_table = split_summary(mock_pidu_combined)
best_ep    = best_val_epoch(mock_pidu_combined)
print(f"Best validation epoch: {best_ep}")

# Highlight the best validation epoch
pidu_table.style \
    .highlight_min(subset=["val"], color="lightgreen") \
    .format("{:.0f}")

## 7. Per-Epoch Aggregation and Statistics

$$\Delta L = \frac{L_0 - L_n}{L_0} \times 100\%$$

In [ ]:
def aggregate_epochs(df: pd.DataFrame) -> pd.DataFrame:
    """Compute per-epoch, per-dataset statistics and relative loss reduction.

    Returns a DataFrame with MultiIndex (dataset, epoch) and columns:
        mean_loss, std_loss, min_loss, max_loss, loss_reduction_pct
    """
    agg = (
        df.groupby(["dataset", "epoch"])["loss"]
        .agg(mean_loss="mean", std_loss="std", min_loss="min", max_loss="max")
        .fillna(0)
    )

    # Compute relative reduction per dataset relative to epoch 0
    def add_reduction(group):
        l0 = group["mean_loss"].iloc[0]
        group["loss_reduction_pct"] = (l0 - group["mean_loss"]) / l0 * 100.0
        return group

    agg = agg.groupby(level="dataset", group_keys=False).apply(add_reduction)
    return agg


pidu_agg = aggregate_epochs(mock_pidu_combined)
print("PIDU aggregation shape:", pidu_agg.shape)
pidu_agg.loc["val"].style.format("{:.1f}").background_gradient(cmap="RdYlGn", subset=["loss_reduction_pct"])

## 8. Model Comparison Utilities

Rank models by final validation loss and compute convergence speed.

A model "converges" at epoch $e^*$ where $L_e < \alpha \cdot L_0$ 
(configurable threshold $\alpha$).

In [ ]:
def convergence_epoch(df: pd.DataFrame, alpha: float = 0.10, split: str = "val") -> int | None:
    """Return the first epoch e where mean loss < alpha * L_0, or None if never reached.

    Parameters
    ----------
    alpha : threshold fraction, e.g. 0.10 means 90% reduction from epoch-0 loss.
    """
    val = df[df["dataset"] == split].groupby("epoch")["loss"].mean()
    l0 = val.iloc[0]
    threshold = alpha * l0
    below = val[val < threshold]
    return int(below.index[0]) if not below.empty else None


def compare_runs(
    run_dict: dict[str, pd.DataFrame],
    split: str = "val",
    alpha: float = 0.10,
) -> pd.DataFrame:
    """Compare multiple models.

    Returns a ranked DataFrame with columns:
        model, final_loss, final_loss_std, final_loss_min, convergence_epoch, rank
    """
    records = []
    for label, df in run_dict.items():
        val_mean = df[df["dataset"] == split].groupby("epoch")["loss"].mean()
        val_std = df[df["dataset"] == split].groupby("epoch")["loss"].std()
        val_min = df[df["dataset"] == split].groupby("epoch")["loss"].min()
        final = float(val_mean.iloc[-1])
        conv_ep = convergence_epoch(df, alpha=alpha, split=split)
        records.append({"model": label, "final_loss": final, "final_loss_std": float(val_std.iloc[-1]), "final_loss_min": float(val_min.iloc[-1]), "convergence_epoch": conv_ep})

    comparison = pd.DataFrame(records).sort_values("final_loss").reset_index(drop=True)
    comparison["rank"] = comparison.index + 1
    return comparison


def plot_final_loss_bar(comparison: pd.DataFrame, set_name:str = "") -> tuple:
    """Bar chart of final validation loss per model."""
    # Make plotting robust in environments without a LaTeX installation.
    plt.rcParams["text.usetex"] = False

    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar(
        comparison["model"],
        comparison["final_loss"],
        color=plt.rcParams["axes.prop_cycle"].by_key()["color"][:len(comparison)]
    )
    ax.set(
        ylabel="Final Loss",
        title=f"Model Comparison - Final Loss {set_name}",
    )
    # x-ticks rotated for readability if many models
    fig.autofmt_xdate(rotation=45, ha="right")
    fig.tight_layout()
    return fig, ax

In [ ]:
# ── Demo ──────────────────────────────────────────────────────────────────────
run_dict = {
    "PIDU": mock_pidu_combined,
    "CNN":  mock_cnn_combined,
    "DU":   mock_du_run0,
}

comparison = compare_runs(run_dict, split="val", alpha=0.10)
print(comparison)

fig, ax = plot_final_loss_bar(comparison)
plt.show()

## 9. Unit Tests — Aggregation and Comparison

In [ ]:
class TestAggregateEpochs(unittest.TestCase):

    def setUp(self):
        self.df = make_mock_run(n_epochs=5, seed=0, start_loss=1000.0, decay=0.80)

    def test_output_shape(self):
        agg = aggregate_epochs(self.df)
        # 4 datasets x 5 epochs
        self.assertEqual(len(agg), 4 * 5)

    def test_required_columns(self):
        agg = aggregate_epochs(self.df)
        for col in ["mean_loss", "std_loss", "min_loss", "max_loss", "loss_reduction_pct"]:
            self.assertIn(col, agg.columns)

    def test_loss_reduction_range(self):
        agg = aggregate_epochs(self.df)
        reductions = agg.loc["train"]["loss_reduction_pct"]
        self.assertTrue((reductions >= 0).all(),
                        "Reduction should be non-negative with decaying loss")
        self.assertTrue((reductions <= 100).all())

    def test_epoch0_reduction_is_zero(self):
        agg = aggregate_epochs(self.df)
        for ds in DATASET_TYPES:
            self.assertAlmostEqual(agg.loc[(ds, 0), "loss_reduction_pct"], 0.0, places=5)


class TestCompareRuns(unittest.TestCase):

    def setUp(self):
        # CNN decays faster (lower alpha=0.5) than PIDU (alpha=0.8)
        self.fast = make_mock_run(n_epochs=10, seed=1, start_loss=50_000.0, decay=0.50)
        self.slow = make_mock_run(n_epochs=10, seed=2, start_loss=50_000.0, decay=0.80)

    def test_ranking_order(self):
        comp = compare_runs({"fast": self.fast, "slow": self.slow})
        self.assertEqual(comp.iloc[0]["model"], "fast",
                         "Model with lower final loss should rank first")

    def test_convergence_fast_before_slow(self):
        fast_ep = convergence_epoch(self.fast, alpha=0.15)
        slow_ep = convergence_epoch(self.slow, alpha=0.15)
        if fast_ep is not None and slow_ep is not None:
            self.assertLessEqual(fast_ep, slow_ep)

    def test_convergence_none_for_high_threshold(self):
        # alpha=0.001 means 99.9% reduction, very unlikely to achieve
        ep = convergence_epoch(self.slow, alpha=0.001)
        self.assertIsNone(ep)


suite2 = unittest.TestLoader().loadTestsFromTestCase(TestAggregateEpochs)
suite3 = unittest.TestLoader().loadTestsFromTestCase(TestCompareRuns)
combined = unittest.TestSuite([suite2, suite3])
runner = unittest.TextTestRunner(verbosity=2)
runner.run(combined)

## 10. Interactive Analysis Template

**Instructions:**
1. Set `TRAILS` and `PATTERNS` to match your experiments.
2. Run the loader cell — it will skip any missing files.
3. Inspect the summary table and plots below.
4. Adjust `alpha` in `compare_runs` to match the convergence threshold you care about.

In [ ]:
# ── Step 1: Configure ─────────────────────────────────────────────────────────
USECASE   = 1
NUM_RUNS  = 8
SKIPROWS_CNN = 11
SKIPROWS_PIDU = 14
INCL_EXP  = False # whether to include 'exp' dataset in plots and tables

# PATTERNS: list[str | tuple[str, int]] = [
#     ("loss_CNN", 1, 11),   # trail 020
#     ("loss_DU", 1, 14),   # trail 021
#     ("loss_PIDU", 1, 14),   # trail 022
# ]
PATTERNS = [
# -----
#     ("loss_PIDU_weight_decay_-0p0001_", 5),
#     ("loss_PIDU_weight_decay_-0p001_",  5),
#     ("loss_PIDU_weight_decay_-0p01_",   5),
#     ("loss_PIDU_weight_decay_-0p1_",    5),
#     ("loss_PIDU_weight_decay_-1p0_",    5),
#     ("loss_PIDU_weight_decay_-10p0_",   5),
# -----
    # ("loss_CNN_cnn_size_-0p1_", 6, SKIPROWS_CNN), 
    # ("loss_CNN_cnn_size_-0p25_", 6, SKIPROWS_CNN), 
    # ("loss_CNN_cnn_size_-0p5_", 6, SKIPROWS_CNN), 
    # ("loss_CNN_cnn_size_-1p0_", 6, SKIPROWS_CNN), 
# -----
  ("loss_PIDU_unfolds_-1_", 1, SKIPROWS_PIDU),
  ("loss_PIDU_unfolds_-2_", 1, SKIPROWS_PIDU),
  ("loss_PIDU_unfolds_-3_", 1, SKIPROWS_PIDU),
  ("loss_PIDU_unfolds_-4_", 1, SKIPROWS_PIDU),
  ("loss_PIDU_unfolds_-5_", 1, SKIPROWS_PIDU),
  ("loss_PIDU_unfolds_-6_", 1, SKIPROWS_PIDU),
  ("loss_PIDU_unfolds_-7_", 1, SKIPROWS_PIDU),
  ("loss_PIDU_unfolds_-8_", 1, SKIPROWS_PIDU),
  # ("loss_PIDU_unfolds_-9_", 1, SKIPROWS_PIDU),
  # ("loss_PIDU_unfolds_-10_", 1, SKIPROWS_PIDU),
# -----
  # ("loss_PIDU_weight_decay_-0p0001_", 5, SKIPROWS_PIDU),
  # ("loss_PIDU_weight_decay_-0p001_", 5, SKIPROWS_PIDU),
  # ("loss_PIDU_weight_decay_-0p01_", 5, SKIPROWS_PIDU),
  # ("loss_PIDU_weight_decay_-0p1_", 5, SKIPROWS_PIDU),
  # ("loss_PIDU_weight_decay_-1p0_", 5, SKIPROWS_PIDU),
  # ("loss_PIDU_weight_decay_-10p0_", 5, SKIPROWS_PIDU),
]

GLOBAL_TRAIL = 1  # fallback when PATTERNS entry is a plain string

In [ ]:
# ── Step 2: Load ──────────────────────────────────────────────────────────────
loaded: dict[str, pd.DataFrame] = {}

for entry in PATTERNS:
    if isinstance(entry, tuple):
        pattern, trail, _skiprows = entry
    else:
        pattern, trail = entry, GLOBAL_TRAIL

    label = f"{pattern} (trail {trail:03d})"
    print(f"\nLoading: {label}")
    try:
        combined, _ = load_multiple_runs(pattern, trail, NUM_RUNS, USECASE, _skiprows)
        loaded[label] = combined
        print(f"  → {len(combined)} rows from {combined['run_id'].nunique()} runs")
    except FileNotFoundError as e:
        print(f"  [WARNING] {e} — skipping")

print("\nLoaded models:", list(loaded.keys()))

In [ ]:
# ── Step 3: Aggregate ─────────────────────────────────────────────────────────
# Fallback to mock data when no real files were found
analysis_dict = loaded

agg_all = {label: aggregate_epochs(df) for label, df in analysis_dict.items()}

for label, agg in agg_all.items():
    val_red = agg.loc["val"]["loss_reduction_pct"].iloc[-1]
    print(f"{label:40s}  val-loss reduction: {val_red:.1f}%")
    if INCL_EXP:
        val_exp = agg.loc["exp"]["loss_reduction_pct"].iloc[-1]
        print(f"{'':40s}  exp-loss reduction: {val_exp:.1f}%")

In [ ]:
# ── Step 4: Compare ───────────────────────────────────────────────────────────
%matplotlib qt
matplotlib.rcParams["font.size"] = 16
import warnings
CONVERGENCE_ALPHA = 0.10   # epoch where val loss < 10 % of epoch-0 val loss

with warnings.catch_warnings():
    warnings.simplefilter("always")
    comparison_train = compare_runs(analysis_dict, split="train", alpha=CONVERGENCE_ALPHA)
    comparison_val = compare_runs(analysis_dict, split="val", alpha=CONVERGENCE_ALPHA)
    

print("\nModel ranking:")
print(comparison_val.to_string(index=False))
print("\nModel ranking (train):")
print(comparison_train.to_string(index=False))

with warnings.catch_warnings():
    warnings.simplefilter("always")
    fig, ax = plot_final_loss_bar(comparison_val, "val")
    plt.show()

    if INCL_EXP:
        comparison_exp = compare_runs(analysis_dict, split="exp", alpha=CONVERGENCE_ALPHA)
        print(comparison_exp.to_string(index=False))
        fig, ax = plot_final_loss_bar(comparison_exp, "exp")
        plt.show()

In [ ]:
# -- Step 5: Plot loss curves -------------------------------------------------
%matplotlib
import scienceplots
import matplotlib
import logging

# Re-apply robust text/font settings after style imports.
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)
plt.rcParams["text.usetex"] = False
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["DejaVu Serif", "Times New Roman", "Times", "STIXGeneral"]
plt.rcParams["font.size"] = 2

# Plot in interactive window
plt.ion()

fig, _ = plot_loss_curves(
    dfs=analysis_dict,
    splits=["train", "val"] + (["exp"] if INCL_EXP else []),
    log_scale=True,
    shade_std=True,
    ylim=(1, 10000),
    style=["science", "ieee"],
    figsize=(12, 6),
    fontsize_legend=2,
)
plt.show()

# All splits for the first model in the dict
first_label, first_df = next(iter(analysis_dict.items()))
fig2, _ = plot_all_splits(
  first_df, 
  title=first_label, 
  log_scale=True,
  fontsize_legend=16,
)
plt.show()

In [ ]:
# ── Optional: export per-model aggregation tables ──────────────────────────────

def export_excel(export_path:str, set_name:str = "val", extra_text:str = "") -> None:
    with pd.ExcelWriter(f"{export_path}_{set_name}_{extra_text}.xlsx") as writer:
      for label, agg in agg_all.items():
          safe_label = f"{names_mapping.get(label, label)}"[:31] # 
          agg_ = agg.loc[set_name]
          agg_.to_excel(writer, sheet_name=safe_label)       # 
    
    print(f"Saved to {export_path}_{set_name}_{extra_text}.xlsx")

EXPORT_PATH = os.path.join(_PROJECT_ROOT, "outputs", "analysis_tables")

names_mapping = {
  "loss_PIDU_nowgn_nostruct (trail 020)": "PIDU-nwgn-ns",
  "loss_PIDU_wgn_nostruct (trail 021)": "PIDU-wgn-ns",
  "loss_PIDU_wgn_struct (trail 022)": "PIDU-wgn-s",
  "loss_PIDU_unfolds_-1_ (trail 007)": "PIDU-unfolds-1",
  "loss_PIDU_unfolds_-2_ (trail 007)": "PIDU-unfolds-2",
  "loss_PIDU_unfolds_-3_ (trail 007)": "PIDU-unfolds-3",
  "loss_PIDU_unfolds_-4_ (trail 007)": "PIDU-unfolds-4",
  "loss_PIDU_unfolds_-5_ (trail 007)": "PIDU-unfolds-5",
  "loss_PIDU_unfolds_-6_ (trail 007)": "PIDU-unfolds-6",
  "loss_PIDU_unfolds_-7_ (trail 007)": "PIDU-unfolds-7",
  "loss_PIDU_unfolds_-8_ (trail 007)": "PIDU-unfolds-8",
  "loss_PIDU_unfolds_-9_ (trail 007)": "PIDU-unfolds-9",
  "loss_PIDU_unfolds_-10_ (trail 007)": "PIDU-unfolds-10",
  "loss_PIDU_weight_decay_-0p0001_ (trail 005)": "PIDU-wd-0p0001",
  "loss_PIDU_weight_decay_-0p001_ (trail 005)": "PIDU-wd-0p001",
  "loss_PIDU_weight_decay_-0p01_ (trail 005)": "PIDU-wd-0p01",
  "loss_PIDU_weight_decay_-0p1_ (trail 005)": "PIDU-wd-0p1",
  "loss_PIDU_weight_decay_-1p0_ (trail 005)": "PIDU-wd-1p0",
  "loss_PIDU_weight_decay_-10p0_ (trail 005)": "PIDU-wd-10p0",
}

export_excel(EXPORT_PATH, set_name="val", extra_text="cnn-size-sweep")

print("Tables ready (export commented out — uncomment EXPORT_PATH cell to save).")
for label, agg in agg_all.items():
    print(f"\n{label}")
    print(agg.loc["val"].round(1).to_string())